In [16]:
# pip install virtualenv 
# virtualenv my_env # create a virtual environment named my_env
# source my_env/bin/activate # activate my_env

In [17]:
# # installing necessary packages in my_env
# pip install youtube-transcript-api==1.2.1
# pip install faiss-cpu==1.8.0
# pip install langchain==0.2.6 | tail -n 1
# pip install langchain-community==0.2.6 | tail -n 1
# pip install ibm-watsonx-ai==1.0.10 | tail -n 1
# pip install langchain_ibm==0.1.8 | tail -n 1
# pip install gradio==4.44.1 | tail -n 1
# # Uninstall any existing huggingface_hub first
# python3.11 -m pip uninstall -y huggingface_hub
# # Install a version compatible with gradio 4.44.1
# python3.11 -m pip install huggingface_hub==0.16.4

In [1]:
import re
from youtube_transcript_api import YouTubeTranscriptApi 

In [5]:
def get_video_id(url):    
    # Regex pattern to match YouTube video URLs
    pattern = r'https:\/\/www\.youtube\.com\/watch\?v=([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else None

In [10]:
url = "https://www.youtube.com/watch?v=BrKBkMCN4qE&list=PLbDHdfrj-Jm1tkQiS4zOobaFvEfPXWFAx"
video_id = get_video_id(url)
print(video_id)  # Output: dQw4w9WgXcQ

BrKBkMCN4qE


In [11]:
def get_transcript(url):
    # Extracts the video ID from the URL
    video_id = get_video_id(url)
    
    # Create a YouTubeTranscriptApi() object
    ytt_api = YouTubeTranscriptApi()
    
    # Fetch the list of available transcripts for the given YouTube video
    transcripts = ytt_api.list(video_id)
    
    transcript = ""
    for t in transcripts:
        # Check if the transcript's language is English
        if t.language_code == 'en':
            if t.is_generated:
                # If no transcript has been set yet, use the auto-generated one
                if len(transcript) == 0:
                    transcript = t.fetch()
            else:
                # If a manually created transcript is found, use it (overrides auto-generated)
                transcript = t.fetch()
                break  # Prioritize the manually created transcript, exit the loop
    
    return transcript if transcript else None

In [12]:
transcript = get_transcript(url)

In [15]:
transcript.language, transcript.is_generated

('English', False)

In [13]:
print(transcript)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Every women has her own way to start her day...', start=95.6, duration=2.014), FetchedTranscriptSnippet(text='Anyone who gets active in the morning...', start=97.814, duration=1.044), FetchedTranscriptSnippet(text="It's because they are expecting something...", start=99.2, duration=1.189), FetchedTranscriptSnippet(text='Whether it is pleasant...', start=100.7, duration=1.0), FetchedTranscriptSnippet(text='Or not..!', start=102.1, duration=0.924), FetchedTranscriptSnippet(text='Whoever leaves its house...', start=103.6, duration=1.176), FetchedTranscriptSnippet(text="Even if this person doesn't want to have a peaceful day...", start=105.2, duration=1.194), FetchedTranscriptSnippet(text='Someone else wishes them that..!', start=106.594, duration=1.532), FetchedTranscriptSnippet(text='Because the truth is...', start=108.6, duration=0.809), FetchedTranscriptSnippet(text="Someone's life...", start=109.7, duration=0.909), FetchedTran

In [19]:
def process(transcript):
    # Initialize an empty string to hold the formatted transcript
    txt = ""
    
    # Loop through each entry in the transcript
    for i in transcript:
        try:
            # Append the text and its start time to the output string
            txt += f"Text: {i.text} Start: {i.start}\n"
        except KeyError:
            # If there is an issue accessing 'text' or 'start', skip this entry
            pass
            
    # Return the processed transcript as a single string
    return txt

In [20]:
formatted_transcript = process(transcript)
# Output the processed transcript
print(formatted_transcript)

Text: Every women has her own way to start her day... Start: 95.6
Text: Anyone who gets active in the morning... Start: 97.814
Text: It's because they are expecting something... Start: 99.2
Text: Whether it is pleasant... Start: 100.7
Text: Or not..! Start: 102.1
Text: Whoever leaves its house... Start: 103.6
Text: Even if this person doesn't want to have a peaceful day... Start: 105.2
Text: Someone else wishes them that..! Start: 106.594
Text: Because the truth is... Start: 108.6
Text: Someone's life... Start: 109.7
Text: Can shift at any moment..! Start: 110.809
Text: Here is some water... Start: 119.8
Text: But.. Mame..? Start: 122.8
Text: Yes..! Start: 123.735
Text: Mame, why use a plastic water jug..? Start: 125.6
Text: The last glass one got broken..! Start: 128.6
Text: Mame... You mean 6 of them..? Start: 131.4
Text: All broken..? Start: 133.5
Text: Yes..! Start: 134.751
Text: The issue is not about them being broken... Start: 135.9
Text: But that you let me know when it occurs.

In [27]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings

In [22]:
def chunk_transcript(processed_transcript, chunk_size=200, chunk_overlap=20):
    # Initialize the RecursiveCharacterTextSplitter with specified chunk size and overlap
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # Split the transcript into chunks
    chunks = text_splitter.split_text(processed_transcript)
    return chunks

In [23]:
# Chunking the transcript
chunks = chunk_transcript(formatted_transcript)
# Output the chunks
print(chunks)

["Text: Every women has her own way to start her day... Start: 95.6\nText: Anyone who gets active in the morning... Start: 97.814\nText: It's because they are expecting something... Start: 99.2", "Text: Whether it is pleasant... Start: 100.7\nText: Or not..! Start: 102.1\nText: Whoever leaves its house... Start: 103.6\nText: Even if this person doesn't want to have a peaceful day... Start: 105.2", "Text: Someone else wishes them that..! Start: 106.594\nText: Because the truth is... Start: 108.6\nText: Someone's life... Start: 109.7\nText: Can shift at any moment..! Start: 110.809", 'Text: Here is some water... Start: 119.8\nText: But.. Mame..? Start: 122.8\nText: Yes..! Start: 123.735\nText: Mame, why use a plastic water jug..? Start: 125.6', 'Text: The last glass one got broken..! Start: 128.6\nText: Mame... You mean 6 of them..? Start: 131.4\nText: All broken..? Start: 133.5\nText: Yes..! Start: 134.751', "Text: The issue is not about them being broken... Start: 135.9\nText: But that

In [26]:
llm = ChatOpenAI(model="gpt-4o", temperature=1, max_retries=3, api_key=os.getenv("OPENAI_API_KEY"))

C:\Users\mbial\AppData\Local\Temp\ipykernel_5192\3107836843.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o", temperature=1, max_retries=3, api_key=os.getenv("OPENAI_API_KEY"))


In [28]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\mbial\AppData\Local\Temp\ipykernel_5192\412152783.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [29]:
from langchain_community.vectorstores import FAISS

In [30]:
def create_faiss_index(chunks, embedding_model):
    """
    Create a FAISS index from text chunks using the specified embedding model.
    
    :param chunks: List of text chunks
    :param embedding_model: The embedding model to use
    :return: FAISS index
    """
    # Use the FAISS library to create an index from the provided text chunks
    return FAISS.from_texts(chunks, embedding_model)

In [31]:
def perform_similarity_search(faiss_index, query, k=3):
    """
    Search for specific queries within the embedded transcript using the FAISS index.
    
    :param faiss_index: The FAISS index containing embedded text chunks
    :param query: The text input for the similarity search
    :param k: The number of similar results to return (default is 3)
    :return: List of similar results
    """
    # Perform the similarity search using the FAISS index
    results = faiss_index.similarity_search(query, k=k)
    return results

In [32]:
from langchain.prompts import PromptTemplate

In [33]:
def create_summary_prompt():
    """
    Create a PromptTemplate for summarizing a YouTube video transcript.
    
    :return: PromptTemplate object
    """
    # Define the template for the summary prompt
    template = """
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    You are an AI assistant tasked with summarizing YouTube video transcripts. Provide concise, informative summaries that capture the main points of the video content.

    Instructions:
    1. Summarize the transcript in a single concise paragraph.
    2. Ignore any timestamps in your summary.
    3. Focus on the spoken content (Text) of the video.

    Note: In the transcript, "Text" refers to the spoken words in the video, and "start" indicates the timestamp when that part begins in the video.<|eot_id|><|start_header_id|>user<|end_header_id|>
    Please summarize the following YouTube video transcript:

    {transcript}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
    """
    
    # Create the PromptTemplate object with the defined template
    prompt = PromptTemplate(
        input_variables=["transcript"],
        template=template
    )
    
    return prompt

In [34]:
from langchain.chains import LLMChain

In [35]:
def create_summary_chain(llm, prompt, verbose=True):
    """
    Create an LLMChain for generating summaries.
    
    :param llm: Language model instance
    :param prompt: PromptTemplate instance
    :param verbose: Boolean to enable verbose output (default: True)
    :return: LLMChain instance
    """
    return LLMChain(llm=llm, prompt=prompt, verbose=verbose)

In [36]:
def retrieve(query, faiss_index, k=7):
    """
    Retrieve relevant context from the FAISS index based on the user's query.

    Parameters:
        query (str): The user's query string.
        faiss_index (FAISS): The FAISS index containing the embedded documents.
        k (int, optional): The number of most relevant documents to retrieve (default is 3).

    Returns:
        list: A list of the k most relevant documents (or document chunks).
    """
    relevant_context = faiss_index.similarity_search(query, k=k)
    return relevant_context

In [37]:
from langchain import PromptTemplate

def create_qa_prompt_template():
    """
    Create a PromptTemplate for question answering based on video content.

    Returns:
        PromptTemplate: A PromptTemplate object configured for Q&A tasks.
    """
    
    # Define the template string
    qa_template = """
    You are an expert assistant providing detailed answers based on the following video content.

    Relevant Video Context: {context}

    Based on the above context, please answer the following question:
    Question: {question}
    """

    # Create the PromptTemplate object
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template=qa_template
    )

    return prompt_template

In [38]:
# Creating the Q&A prompt template 
qa_prompt_template = create_qa_prompt_template()

# Example of how to use the prompt template with context and a question
context = "This video is a movie about a man who is cheating on his wife."
question = "What are the key principles discussed in the video?"

# Generating the prompt
generated_prompt = qa_prompt_template.format(context=context, question=question)

# Output the generated prompt
print(generated_prompt)


    You are an expert assistant providing detailed answers based on the following video content.

    Relevant Video Context: This video is a movie about a man who is cheating on his wife.

    Based on the above context, please answer the following question:
    Question: What are the key principles discussed in the video?
    


In [39]:
def create_qa_chain(llm, prompt_template, verbose=True):
    """
    Create an LLMChain for question answering.

    Args:
        llm: Language model instance
            The language model to use in the chain (e.g., WatsonxGranite).
        prompt_template: PromptTemplate
            The prompt template to use for structuring inputs to the language model.
        verbose: bool, optional (default=True)
            Whether to enable verbose output for the chain.

    Returns:
        LLMChain: An instantiated LLMChain ready for question answering.
    """
    
    return LLMChain(llm=llm, prompt=prompt_template, verbose=verbose)

In [40]:
def generate_answer(question, faiss_index, qa_chain, k=7):
    """
    Retrieve relevant context and generate an answer based on user input.

    Args:
        question: str
            The user's question.
        faiss_index: FAISS
            The FAISS index containing the embedded documents.
        qa_chain: LLMChain
            The question-answering chain (LLMChain) to use for generating answers.
        k: int, optional (default=3)
            The number of relevant documents to retrieve.

    Returns:
        str: The generated answer to the user's question.
    """

    # Retrieve relevant context
    relevant_context = retrieve(question, faiss_index, k=k)

    # Generate answer using the QA chain
    answer = qa_chain.predict(context=relevant_context, question=question)

    return answer

In [41]:
# Initialize an empty string to store the processed transcript after fetching and preprocessing
processed_transcript = ""

def summarize_video(video_url):
    """
    Title: Summarize Video

    Description:
    This function generates a summary of the video using the preprocessed transcript.
    If the transcript hasn't been fetched yet, it fetches it first.

    Args:
        video_url (str): The URL of the YouTube video from which the transcript is to be fetched.

    Returns:
        str: The generated summary of the video or a message indicating that no transcript is available.
    """
    global fetched_transcript, processed_transcript
    
    
    if video_url:
        # Fetch and preprocess transcript
        fetched_transcript = get_transcript(video_url)
        processed_transcript = process(fetched_transcript)
    else:
        return "Please provide a valid YouTube URL."

    if processed_transcript:
        

        # Step 3: Create the summary prompt and chain
        summary_prompt = create_summary_prompt()
        summary_chain = create_summary_chain(llm, summary_prompt)

        # Step 4: Generate the video summary
        summary = summary_chain.run({"transcript": processed_transcript})
        return summary
    else:
        return "No transcript available. Please fetch the transcript first."

In [42]:
def answer_question(video_url, user_question):
    """
    Title: Answer User's Question

    Description:
    This function retrieves relevant context from the FAISS index based on the user’s query 
    and generates an answer using the preprocessed transcript.
    If the transcript hasn't been fetched yet, it fetches it first.

    Args:
        video_url (str): The URL of the YouTube video from which the transcript is to be fetched.
        user_question (str): The question posed by the user regarding the video.

    Returns:
        str: The answer to the user's question or a message indicating that the transcript 
             has not been fetched.
    """
    global fetched_transcript, processed_transcript

    # Check if the transcript needs to be fetched
    if not processed_transcript:
        if video_url:
            # Fetch and preprocess transcript
            fetched_transcript = get_transcript(video_url)
            processed_transcript = process(fetched_transcript)
        else:
            return "Please provide a valid YouTube URL."

    if processed_transcript and user_question:
        # Step 1: Chunk the transcript (only for Q&A)
        chunks = chunk_transcript(processed_transcript)

        # Step 2: Set up IBM Watson credentials
        
        faiss_index = create_faiss_index(chunks, embedding_model)

        # Step 5: Set up the Q&A prompt and chain
        qa_prompt = create_qa_prompt_template()
        qa_chain = create_qa_chain(llm, qa_prompt)

        # Step 6: Generate the answer using FAISS index
        answer = generate_answer(user_question, faiss_index, qa_chain)
        return answer
    else:
        return "Please provide a valid question and ensure the transcript has been fetched."

In [43]:
import gradio as gr

In [ ]:
with gr.Blocks() as interface:
    # Input field for YouTube URL
    video_url = gr.Textbox(label="YouTube Video URL", placeholder="Enter the YouTube Video URL")
    
    # Outputs for summary and answer
    summary_output = gr.Textbox(label="Video Summary", lines=5)
    question_input = gr.Textbox(label="Ask a Question About the Video", placeholder="Ask your question")
    answer_output = gr.Textbox(label="Answer to Your Question", lines=5)

    # Buttons for selecting functionalities after fetching transcript
    summarize_btn = gr.Button("Summarize Video")
    question_btn = gr.Button("Ask a Question")

    # Display status message for transcript fetch
    transcript_status = gr.Textbox(label="Transcript Status", interactive=False)

    # Set up button actions
    summarize_btn.click(summarize_video, inputs=video_url, outputs=summary_output)
    question_btn.click(answer_question, inputs=[video_url, question_input], outputs=answer_output)

# Launch the app with specified server name and port
interface.launch(server_name="0.0.0.0", server_port=7860)

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\mbial\AppData\Local\Temp\ipykernel_5192\88981817.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  return LLMChain(llm=llm, prompt=prompt, verbose=verbose)
C:\Users\mbial\AppData\Local\Temp\ipykernel_5192\1711515002.py:36: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  summary = summary_chain.run({"transcript": processed_transcript})




> Entering new LLMChain chain...
Prompt after formatting:

    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    You are an AI assistant tasked with summarizing YouTube video transcripts. Provide concise, informative summaries that capture the main points of the video content.

    Instructions:
    1. Summarize the transcript in a single concise paragraph.
    2. Ignore any timestamps in your summary.
    3. Focus on the spoken content (Text) of the video.

    Note: In the transcript, "Text" refers to the spoken words in the video, and "start" indicates the timestamp when that part begins in the video.<|eot_id|><|start_header_id|>user<|end_header_id|>
    Please summarize the following YouTube video transcript:

    Text: Every women has her own way to start her day... Start: 95.6
Text: Anyone who gets active in the morning... Start: 97.814
Text: It's because they are expecting something... Start: 99.2
Text: Whether it is pleasant... Start: 100.7
Text: Or not..! Start